EXOPLANET HABITABILITY CLASSIFICATION END-TO-END PROJECT

In [45]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as pyplot

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectFromModel

#Machine Learning Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn import tree
from sklearn.naive_bayes import GaussianNB

In [46]:
df = pd.read_csv("phl_exoplanet_catalog_2019.csv")
df.head()

,P_NAME,P_STATUS,P_MASS,P_MASS_ERROR_MIN,P_MASS_ERROR_MAX,P_RADIUS,P_RADIUS_ERROR_MIN,P_RADIUS_ERROR_MAX,P_YEAR,P_UPDATED,...,P_HABZONE_CON,P_TYPE_TEMP,P_HABITABLE,P_ESI,S_CONSTELLATION,S_CONSTELLATION_ABR,S_CONSTELLATION_ENG,P_RADIUS_EST,P_MASS_EST,P_SEMI_MAJOR_AXIS_EST
0,11 Com b,3.0,6165.86330,-476.74200,476.74200,NaN,NaN,NaN,2007,2014-05-14,...,0,Hot,0,0.083813,Coma Berenices,Com,Berenice's Hair,12.082709,6165.86330,1.29
1,11 UMi b,3.0,4684.78480,-794.57001,794.57001,NaN,NaN,NaN,2009,2018-09-06,...,0,Hot,0,0.082414,Ursa Minor,UMi,Little Bear,12.229641,4684.78480,1.53
2,14 And b,3.0,1525.57440,NaN,NaN,NaN,NaN,NaN,2008,2014-05-14,...,0,Hot,0,0.081917,Andromeda,And,Andromeda,12.848516,1525.57440,0.83
3,14 Her b,3.0,1481.07850,-47.67420,47.67420,NaN,NaN,NaN,2002,2018-09-06,...,0,Cold,0,0.145241,Hercules,Her,Hercules,12.865261,1481.07850,2.93
4,16 Cyg B b,3.0,565.73385,-25.42624,25.42624,NaN,NaN,NaN,1996,2018-09-06,...,1,Warm,0,0.368627,Cygnus,Cyg,Swan,13.421749,565.73385,1.66


In [42]:
df.columns

Index(['P_NAME', 'P_STATUS', 'P_MASS', 'P_MASS_ERROR_MIN', 'P_MASS_ERROR_MAX',
       'P_RADIUS', 'P_RADIUS_ERROR_MIN', 'P_RADIUS_ERROR_MAX', 'P_YEAR',
       'P_UPDATED',
       ...
       'P_HABZONE_CON', 'P_TYPE_TEMP', 'P_HABITABLE', 'P_ESI',
       'S_CONSTELLATION', 'S_CONSTELLATION_ABR', 'S_CONSTELLATION_ENG',
       'P_RADIUS_EST', 'P_MASS_EST', 'P_SEMI_MAJOR_AXIS_EST'],
      dtype='object', length=112)

In [43]:
shape = df.shape
print(f'Rows: {shape[0]}')
print(f'Columns: {shape[1]}')

Rows: 4048
Columns: 112


In [51]:
null_vales = df.isnull().sum().sort_values(ascending = False)
null_vales.head(40
                )

P_ATMOSPHERE                    4048
P_ALT_NAMES                     4048
P_DETECTION_RADIUS              4048
P_GEO_ALBEDO                    4048
P_DETECTION_MASS                4048
S_MAGNETIC_FIELD                4048
S_DISC                          4048
P_TEMP_MEASURED                 4043
P_GEO_ALBEDO_ERROR_MIN          4043
P_GEO_ALBEDO_ERROR_MAX          4043
P_TPERI_ERROR_MAX               3576
P_TPERI_ERROR_MIN               3576
P_TPERI                         3567
P_OMEGA_ERROR_MIN               3355
P_OMEGA_ERROR_MAX               3355
P_ESCAPE                        3342
P_POTENTIAL                     3342
P_DENSITY                       3342
P_GRAVITY                       3342
P_OMEGA                         3302
P_INCLINATION_ERROR_MAX         3238
P_INCLINATION_ERROR_MIN         3236
P_INCLINATION                   3204
P_ECCENTRICITY_ERROR_MIN        3077
P_ECCENTRICITY_ERROR_MAX        3077
S_TYPE                          2678
P_ECCENTRICITY                  2668
P

In [ ]:
df = df.dropna(["P_DETECTION_MASS", "P_GEO_ALBEDO", "S_MAGNETIC_FIELD", "S_DISC", "P_ATMOSPHERE", "P_ALT_NAMES", "P_DETECTION_RADIUS", "P_GEO_ALBEDO_ERROR_MIN", "P_TEMP_MEASURED", "P_GEO_ALBEDO_ERROR_MAX","P_TPERI_ERROR_MAX", "P_TPERI_ERROR_MIN", "P_TPERI", "P_OMEGA_ERROR_MIN", "P_OMEGA_ERROR_MAX", "P_DENSITY", "P_ESCAPE", "P_POTENTIAL", "P_GRAVITY", "P_OMEGA",   "P_INCLINATION_ERROR_MAX", "P_INCLINATION_ERROR_MIN", "P_INCLINATION", "P_ECCENTRICITY_ERROR_MAX", "P_ECCENTRICITY_ERROR_MIN", "S_TYPE", "P_ECCENTRICITY","P_IMPACT_PARAMETER_ERROR_MIN", "P_IMPACT_PARAMETER_ERROR_MAX", "P_IMPACT_PARAMETER", "P_MASS_ERROR_MAX", "P_MASS_ERROR_MIN", "P_HILL_SPHERE", "P_SEMI_MAJOR_AXIS_ERROR_MIN", "P_SEMI_MAJOR_AXIS_ERROR_MAX", "P_MASS"], axis=1)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4048 entries, 0 to 4047
Data columns (total 76 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   P_NAME                   4048 non-null   object 
 1   P_STATUS                 4048 non-null   float64
 2   P_RADIUS                 3139 non-null   float64
 3   P_RADIUS_ERROR_MIN       3105 non-null   float64
 4   P_RADIUS_ERROR_MAX       3105 non-null   float64
 5   P_YEAR                   4048 non-null   int64  
 6   P_UPDATED                4048 non-null   object 
 7   P_PERIOD                 3938 non-null   float64
 8   P_PERIOD_ERROR_MIN       3807 non-null   float64
 9   P_PERIOD_ERROR_MAX       3807 non-null   float64
 10  P_SEMI_MAJOR_AXIS        2367 non-null   float64
 11  P_ANGULAR_DISTANCE       2361 non-null   float64
 12  P_DETECTION              4048 non-null   object 
 13  S_NAME                   4048 non-null   object 
 14  S_RA                    

In [60]:
print(df['P_TYPE'])
df['P_TYPE'].isnull().sum()

0            Jovian
1            Jovian
2            Jovian
3            Jovian
4            Jovian
           ...     
4043    Superterran
4044      Neptunian
4045         Terran
4046         Terran
4047         Terran
Name: P_TYPE, Length: 4048, dtype: object


np.int64(17)

The “P_TYPE” column in our data describes the exoplanet category and consists of six possible values: Jovian, Superterran, Neptunian, Terran, Subterran, and Miniterran. This feature had 17 missing values.

TO handle the missing values, we can impute them using mode

In [61]:
df['P_TYPE'] = df['P_TYPE'].fillna(df['P_TYPE'].mode()[0]) #set the first most common value
df['P_TYPE_TEMP'] = df['P_TYPE_TEMP'].fillna(df['P_TYPE_TEMP'].mode()[0])
df['S_TYPE_TEMP'] = df['S_TYPE_TEMP'].fillna(df['S_TYPE_TEMP'].mode()[0]) 

nOw WE HANDLED MISSING VALUES IN CATEGORICAL DATA, NO WE WILL WORK UPON NUMERICAL DATA 

We are going to use Scikit-learn's imputer because it provides a systematic way to replace missing numerical data, and IterativeImputer is particularly useful when we want to use relationships between multiple features to make those replacements.


In [68]:
numeric_cols = df.select_dtypes(include='number').columns

# Impute only numeric columns
imp = IterativeImputer(max_iter=10, verbose=0)
df[numeric_cols] = imp.fit_transform(df[numeric_cols])

we have corrected all of our missing values! But, we still have a major class imbalance to work with. Since we don’t know of many potentially habitable planets, the number of inhabitable exoplanets greatly outnumbers the habitable ones. This will likely cause our model to guess an exoplanet is inhabitable, even if it’s not the case, because it will still have an average of like 99% accuracy. Thus, we need to find a way to balance the scales.

We will use upsampling method, using algorithm SMOTE.

In [76]:
X= df.iloc[:, df.columns != 'P_HABITABLE']
y = df['P_HABITABLE']
smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)
X_res, y_res = smote.fit_resample(X,y)

df = pd.concat([pd.DataFrame(X_res), pd.DataFrame(y_res)], axis = 1 )
df.head()

,P_STATUS,P_RADIUS,P_RADIUS_ERROR_MIN,P_RADIUS_ERROR_MAX,P_YEAR,P_PERIOD,P_PERIOD_ERROR_MIN,P_PERIOD_ERROR_MAX,P_SEMI_MAJOR_AXIS,P_ANGULAR_DISTANCE,...,S_SNOW_LINE,S_ABIO_ZONE,S_TIDAL_LOCK,P_HABZONE_OPT,P_HABZONE_CON,P_ESI,P_RADIUS_EST,P_MASS_EST,P_SEMI_MAJOR_AXIS_EST,P_HABITABLE
0,3.0,4.794721,-0.433851,0.608203,2007.0,326.03000,-0.32,0.32,1.29,13.8,...,34.529063,0.476460,0.642400,0.0,0.0,0.083813,12.082709,6165.86330,1.29,0.0
1,3.0,4.794721,-0.433851,0.608203,2009.0,516.21997,-3.20,3.20,1.53,12.2,...,42.732816,0.193891,0.648683,0.0,0.0,0.082414,12.229641,4684.78480,1.53,0.0
2,3.0,4.794721,-0.433851,0.608203,2008.0,185.84000,-0.23,0.23,0.83,11.0,...,20.593611,0.502752,0.600010,0.0,0.0,0.081917,12.848516,1525.57440,0.83,0.0
3,3.0,4.794721,-0.433851,0.608203,2002.0,1773.40000,-2.50,2.50,2.93,163.0,...,2.141648,0.919756,0.445415,0.0,0.0,0.145241,12.865261,1481.07850,2.93,0.0
4,3.0,4.794721,-0.433851,0.608203,1996.0,798.50000,-1.00,1.00,1.66,78.5,...,3.019411,1.983658,0.473325,1.0,1.0,0.368627,13.421749,565.73385,1.66,0.0
